## Model Monitoring Setup
Data Capture, Baseline Generation, Deployment Preparation for Model Quality Monitoring

In [18]:
# Imports & Setup
import boto3
import pandas as pd
import numpy as np
import json
import botocore
from botocore.exceptions import ClientError

import sagemaker
from sagemaker import get_execution_role, Session
from sagemaker.model_monitor import DataCaptureConfig, DefaultModelMonitor
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.model_monitor import CronExpressionGenerator

In [8]:
# Initialize session and role
session = Session()
role = get_execution_role()
region = session.boto_region_name
s3 = boto3.client('s3')
sagemaker_client = boto3.client('sagemaker', region_name=region)

In [9]:
# Load your engineered dataset
df = pd.read_csv('cardio_engineered.csv')

# Display first few rows to verify
print(df.head())

# Define cleaned file name
clean_file = 'cardio_engineered_clean.csv'

# Save cleaned CSV locally
df.to_csv(clean_file, index=False, encoding='utf-8-sig')
print(f"Saved cleaned dataset locally as '{clean_file}'")

# Define S3 upload configuration
bucket = 'sagemaker-us-east-1-226675648827'
prefix = 'cardio_data'
s3_key = f'{prefix}/{clean_file}'

# Initialize S3 client
s3_client = boto3.client('s3')

# Upload to S3
try:
    s3_client.upload_file(clean_file, bucket, s3_key)
    print(f"Uploaded '{clean_file}' to s3://{bucket}/{s3_key}")
except Exception as e:
    print(f"Upload failed: {e}")

   age  gender  height_ft  weight_lbs  systolic_bp  diastolic_bp  cholesterol  \
0   50       2       5.51      136.69          110            80            1   
1   55       1       5.12      187.39          140            90            3   
2   51       1       5.41      141.10          130            70            3   
3   48       2       5.54      180.78          150           100            1   
4   47       1       5.12      123.46          100            60            1   

   gluc  smoke  alco  ...  cholesterol_label  pulse_pressure  chol_bmi_ratio  \
0     1      0     0  ...             Normal              30            4.55   
1     1      0     0  ...  Well Above Normal              50            8.60   
2     1      0     0  ...  Well Above Normal              60           12.74   
3     1      0     0  ...             Normal              50            3.48   
4     1      0     0  ...             Normal              40            4.35   

  height_in age_years  is_hypert

### Logistic Regression Endpoint

In [14]:
# Define model artifact and inference script
model_artifact = 's3://sagemaker-us-east-1-226675648827/model/logistic/logistic_model.tar.gz'
entry_point = 'inference.py'
endpoint_name = 'cardio-logistic-monitor-endpoint'

# Create SKLearn model object
sklearn_model = SKLearnModel(
    model_data=model_artifact,
    role=role,
    entry_point=entry_point,
    framework_version='0.23-1',
    sagemaker_session=session
)

# Enable full data capture configuration
data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=f's3://{bucket}/data-capture/logistic',
    capture_options=['Request', 'Response']
)

# Deploy if endpoint
sagemaker_client = boto3.client('sagemaker', region_name=region)

def deploy_if_not_exists(model, endpoint_name, instance_type, data_capture_config):
    try:
        sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
        print(f"Endpoint '{endpoint_name}' already exists. Skipping deployment.")
    except sagemaker_client.exceptions.ClientError as e:
        if 'Could not find endpoint' in str(e):
            print(f"Deploying model to new endpoint '{endpoint_name}'...")
            model.deploy(
                initial_instance_count=1,
                instance_type=instance_type,
                endpoint_name=endpoint_name,
                data_capture_config=data_capture_config
            )
            print(f"Deployed endpoint '{endpoint_name}' successfully.")
        else:
            raise

# Deploy the model
deploy_if_not_exists(
    model=sklearn_model,
    endpoint_name=endpoint_name,
    instance_type='ml.m5.xlarge',
    data_capture_config=data_capture_config
)

Deploying model to new endpoint 'cardio-logistic-monitor-endpoint'...
------!Deployed endpoint 'cardio-logistic-monitor-endpoint' successfully.


### Random Forest Endpoint

In [17]:
# Define Random Forest model artifact and endpoint name
rf_model_artifact = 's3://sagemaker-us-east-1-226675648827/model/random_forest/random_forest_model.tar.gz'
rf_entry_point = 'inference_rf.py'
rf_endpoint_name = 'cardio-rf-monitor-endpoint'

# Create the SKLearnModel object for Random Forest
rf_model = SKLearnModel(
    model_data=rf_model_artifact,
    role=role,
    entry_point=rf_entry_point,
    framework_version='0.23-1',
    sagemaker_session=session
)

# Data capture configuration for Random Forest endpoint
rf_data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=f's3://{bucket}/data-capture/random-forest',
    capture_options=['Request', 'Response']
)

# Deploy if the endpoint doesn't exist
def deploy_rf_if_not_exists(model, endpoint_name, instance_type, data_capture_config):
    try:
        sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
        print(f"Endpoint '{endpoint_name}' already exists. Skipping deployment.")
    except sagemaker_client.exceptions.ClientError as e:
        if 'Could not find endpoint' in str(e):
            print(f"Deploying Random Forest model to endpoint '{endpoint_name}'...")
            model.deploy(
                initial_instance_count=1,
                instance_type=instance_type,
                endpoint_name=endpoint_name,
                data_capture_config=data_capture_config
            )
            print(f"Deployed endpoint '{endpoint_name}' successfully.")
        else:
            raise

# Trigger deployment for Random Forest model
deploy_rf_if_not_exists(
    model=rf_model,
    endpoint_name=rf_endpoint_name,
    instance_type='ml.m5.xlarge',
    data_capture_config=rf_data_capture_config
)

Deploying Random Forest model to endpoint 'cardio-rf-monitor-endpoint'...
------!Deployed endpoint 'cardio-rf-monitor-endpoint' successfully.


### Generate Baseline statistics.json and contraints.json

In [20]:
# Create a monitor instance
monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
    sagemaker_session=session
)

# Define S3 locations
bucket = 'sagemaker-us-east-1-226675648827'
prefix = 'cardio_data'
baseline_data_uri = f's3://{bucket}/{prefix}/cardio_engineered_clean.csv'
baseline_results_uri = f's3://{bucket}/{prefix}/baseline-results'

# Suggest a baseline
baseline_job = monitor.suggest_baseline(
    baseline_dataset=baseline_data_uri,
    dataset_format={'csv': {'header': True}},
    output_s3_uri=baseline_results_uri,
    wait=True
)

print("Baseline generation complete")
print(f"statistics.json: {baseline_results_uri}/statistics.json")
print(f"constraints.json: {baseline_results_uri}/constraints.json")


Job Name:  baseline-suggestion-job-2025-06-15-23-58-15-556
Inputs:  [{'InputName': 'baseline_dataset_input', 'AppManaged': False, 'S3Input': {'S3Uri': 's3://sagemaker-us-east-1-226675648827/cardio_data/cardio_engineered_clean.csv', 'LocalPath': '/opt/ml/processing/input/baseline_dataset_input', 'S3DataType': 'S3Prefix', 'S3InputMode': 'File', 'S3DataDistributionType': 'FullyReplicated', 'S3CompressionType': 'None'}}]
Outputs:  [{'OutputName': 'monitoring_output', 'AppManaged': False, 'S3Output': {'S3Uri': 's3://sagemaker-us-east-1-226675648827/cardio_data/baseline-results', 'LocalPath': '/opt/ml/processing/output', 'S3UploadMode': 'EndOfJob'}}]
..............2025-06-16 00:00:24.040539: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-06-16 00:00:24.040570: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart d

### Create Monitor & Schedule
#### Logistic Regression

In [25]:
# Baseline files
baseline_statistics_uri = f"s3://{bucket}/cardio_data/baseline-results/statistics.json"
baseline_constraints_uri = f"s3://{bucket}/cardio_data/baseline-results/constraints.json"

In [26]:
# Monitor instance for Logistic Regression
log_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
    sagemaker_session=session
)

log_monitor.create_monitoring_schedule(
    monitor_schedule_name="cardio-logistic-monitor-schedule",
    endpoint_input="cardio-logistic-monitor-endpoint",
    output_s3_uri=f"s3://{bucket}/monitoring/logistic/monitor-output",
    statistics=baseline_statistics_uri,
    constraints=baseline_constraints_uri,
    enable_cloudwatch_metrics=True,
    schedule_cron_expression=CronExpressionGenerator.hourly()
)

print("Logistic Regression monitoring schedule created.")

Logistic Regression monitoring schedule created.


#### Random Forest

In [28]:
# Create monitor object for Random Forest
rf_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
    sagemaker_session=session
)

# Create monitoring schedule for Random Forest endpoint
rf_monitor.create_monitoring_schedule(
    monitor_schedule_name="cardio-rf-monitor-schedule",
    endpoint_input="cardio-rf-monitor-endpoint",
    output_s3_uri=f"s3://{bucket}/monitoring/random-forest/monitor-output",
    statistics=baseline_statistics_uri,
    constraints=baseline_constraints_uri,
    enable_cloudwatch_metrics=True,
    schedule_cron_expression=CronExpressionGenerator.hourly()  # or .daily()
)

print("Random Forest monitoring schedule created.")

Random Forest monitoring schedule created.


* Calculates <b>statistics</b> (distribution, min, max, mean, std, percentiles, etc.)
* Generates <b>constraints</b> (rules/thresholds learned from your dataset, e.g., feature X must be within certain boundaries)

### Check Constraint Files Exist

In [31]:
baseline_prefix = 'cardio_data/baseline-results/'
expected_files = ['statistics.json', 'constraints.json']

# Check if both files exist in the S3 path
response = s3_client.list_objects_v2(Bucket=bucket, Prefix=baseline_prefix)

found_files = [obj['Key'].split('/')[-1] for obj in response.get('Contents', [])]

for expected_file in expected_files:
    if expected_file in found_files:
        print(f"{expected_file} found")
    else:
        print(f"{expected_file} MISSING")

statistics.json found
constraints.json found


In [32]:
# List all objects in the bucket
response = s3.list_objects_v2(Bucket=bucket)

print("Files containing 'baseline':\n")

if 'Contents' in response:
    for obj in response['Contents']:
        key = obj['Key']
        if 'baseline' in key.lower():
            print(f"• {key}")
else:
    print("No objects found in the bucket.")

Files containing 'baseline':

• cardio_data/baseline-results/constraints.json
• cardio_data/baseline-results/statistics.json
• cardio_project/cardio_logistic_baseline_v2.ipynb
• model/cardio_logistic_baseline_v2.ipynb


In [33]:
# Upload
s3_client.upload_file(clean_file, bucket, f'{prefix}/cardio_engineered_clean.csv')
print("Cleaned CSV saved to S3.")

Cleaned CSV saved to S3.


In [35]:
!aws s3 cp cardio_monitoring_endpoint_scheduling.ipynb s3://sagemaker-us-east-1-226675648827/monitoring/cardio_monitoring_endpoint_scheduling.ipynb

upload: ./cardio_monitoring_endpoint_scheduling.ipynb to s3://sagemaker-us-east-1-226675648827/monitoring/cardio_monitoring_endpoint_scheduling.ipynb
